# Batching equivariant matrices

This notebook introduces you to one aspect of generating matrices that you will inevitably face when training a model: **batching**.

Prerequisites
-------------

Before reading this notebook, **make sure you have read the [notebook on computing a matrix](<./Computing a matrix.ipynb>)**, which introduces all of the most basic concepts of `graph2mat` that we are going to assume are already known. Also **we will use exactly the same setup**, with the only difference that we will compute **two matrices at the same time instead of just one**.

In [1]:
import os
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

In [2]:
import numpy as np

# So that we can plot sisl geometries
import sisl.viz

from e3nn import o3

from graph2mat import (
    PointBasis,
    BasisTableWithEdges,
    BasisConfiguration,
    MatrixDataProcessor,
)

from graph2mat.bindings.torch import TorchBasisMatrixData, TorchBasisMatrixDataset
from graph2mat.bindings.e3nn import E3nnGraph2Mat

from graph2mat.tools.viz import plot_basis_matrix

/home/ICN2/snavarro/.local/lib/python3.10/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))
/home/ICN2/snavarro/.local/lib/python3.10/site-packages/torch/__config__.py:9: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 11040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:109.)
  return torch._C._show_config()


The matrix-computing function 
-----------------------------

As we have already seen in the notebook on computing a matrix, we need to define a **basis**, **a basis table**, **a data processor** and **the shape of the node features**. With all this, we can **initialize the matrix-computing function**. We define everything exactly as in the other notebook:

In [3]:
# The basis
point_1 = PointBasis("A", R=2, basis="0e", basis_convention="spherical")
point_2 = PointBasis("B", R=5, basis="2x0e + 1o", basis_convention="spherical")

basis = [point_1, point_2]

# The basis table.
table = BasisTableWithEdges(basis)

# The data processor.
processor = MatrixDataProcessor(
    basis_table=table, symmetric_matrix=False, sub_point_matrix=False
)

# The shape of the node features.
node_feats_irreps = o3.Irreps("0e + 1o")


# The fake environment representation function that we will use
# to compute node features.
def get_environment_representation(data, irreps):
    """Function that mocks a true calculation of an environment representation.

    Computes a random array and then ensures that the numbers obey our particular
    system's symmetries.
    """

    node_features = irreps.randn(data.num_nodes, -1)
    # The point in the middle sees the same in -X and +X directions
    # therefore its representation must be 0.
    # In principle the +/- YZ are also equivalent, but let's say that there
    # is something breaking the symmetry to make the numbers more interesting.
    # Note that the spherical harmonics convention is YZX.
    node_features[1, 3] = 0
    # We make both A points have equivalent features except in the X direction,
    # where the features are opposite
    node_features[2::3, :3] = node_features[0::3, :3]
    node_features[2::3, 3] = -node_features[0::3, 3]
    return node_features


# The matrix readout function
model = E3nnGraph2Mat(
    unique_basis=basis,
    irreps=dict(node_feats_irreps=node_feats_irreps),
    symmetric=False,
    preprocessing_edges=None,
)

BasisTableWithEdges: is_square = True
BasisTableWithEdges: all matrix roles = [None, None]
Basis sizes: [1 5]
Basis sizes: [1 5]
In BasisTableWithEdges: 
self.edge_type == point_types_to_edge_types:
[[ 0  1]
 [-1  2]]
self.point_block_shape:
[[1 5]
 [1 5]]
self.point_block_size:
[ 1 25]
Point type to edge type:
[[ 0  1]
 [-1  2]]
Edge type to point types:
[[0 0]
 [0 1]
 [1 1]]
Edge block shape:
[[1 1 5]
 [1 5 5]]
Edge block size:
[ 1  5 25]
BasisTableWithEdges: is_square = True
BasisTableWithEdges: all matrix roles = [None, None]
Basis sizes: [1 5]
Basis sizes: [1 5]
In BasisTableWithEdges: 
self.edge_type == point_types_to_edge_types:
[[ 0  1]
 [-1  2]]
self.point_block_shape:
[[1 5]
 [1 5]]
self.point_block_size:
[ 1 25]
Point type to edge type:
[[ 0  1]
 [-1  2]]
Edge type to point types:
[[0 0]
 [0 1]
 [1 1]]
Edge block shape:
[[1 1 5]
 [1 5 5]]
Edge block size:
[ 1  5 25]


/home/ICN2/snavarro/.local/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
/home/ICN2/snavarro/.local/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
/home/ICN2/snavarro/.local/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
/home/ICN2/snavarro

Creating two configurations
---------------------------

Now, **we will create two configurations instead of one**. Both will have the same coordinates, the only difference will be that **we will swap the point types**. However, you could give different coordinates to each of them as well, or a different number of atoms.

We'll store both configurations in a `configs` list.

In [4]:
positions = np.array([[0, 0, 0], [6.0, 0, 0], [12, 0, 0]])

config1 = BasisConfiguration(
    point_types=["A", "B", "A"],
    positions=positions,
    basis=basis,
    cell=np.eye(3) * 100,
    pbc=(False, False, False),
)

config2 = BasisConfiguration(
    point_types=["B", "A", "B"],
    positions=positions,
    basis=basis,
    cell=np.eye(3) * 100,
    pbc=(False, False, False),
)

configs = [config1, config2]

As we did in the other notebook, we plot the configurations to see how they look like, and visualize the overlaps:

In [5]:
geom1 = config1.to_sisl_geometry()
geom1.plot(show_cell=False, atoms_style={"size": geom1.maxR(all=True)}).update_layout(
    title="Config 1"
).show()

geom2 = config2.to_sisl_geometry()
geom2.plot(show_cell=False, atoms_style={"size": geom2.maxR(all=True)}).update_layout(
    title="Config 2"
).show()

Build a dataset
---------------

With all our configurations, we can **create a dataset**. The specific class that does this is the `TorchBasisMatrixDataset`, which **apart from the configurations needs the data processor** as usual. 

In [6]:
dataset = TorchBasisMatrixDataset(configs, data_processor=processor)

This dataset contains all the configurations. We now just need some tool to create batches from it.

Batching with a DataLoader
-------------------

`TorchBasisMatrixDataset` is just an extension of `torch.utils.data.Dataset`. Therefore, you don't need a `graph2mat` specific tool to create batches. In fact, **we recommend that you use** `torch_geometric`'s `DataLoader`:

In [7]:
from torch_geometric.loader import DataLoader

Everything that you need to do is: **pass the dataset** and **specify some batch size**. 

In [8]:
loader = DataLoader(dataset, batch_size=2)

In this case we use a batch size of `2`, which is our total number of configurations. Therefore, **we will only have one batch**.

Let's loop through the batches (only 1) and print them:

In [9]:
for data in loader:
    print(data)

self.basis_table.R is an array: [2 5]
point_types: [0 1 0]
self.basis_table.R[point_types]: [2 5 2]
Cutoff: [1.9999 4.9999 1.9999]
In BasisMatrixData.from_config 1:
edge_index: [[0 1 1 2]
 [1 0 2 1]]
edge_types: [ 1 -1 -1  1]
In sort_edge_index:
isc_off: [[[0]]]
sc_shifts: [[0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]]
isc: [0 0 0 0]
In BasisMatrixData.from_config 2:
edge_index: [[0 1 2 1]
 [1 0 1 2]]
edge_types: [ 1 -1  1 -1]
In BasisMatrixData.from_config 3:
edge_index: [[0 1 2 1]
 [1 0 1 2]]
edge_types: [ 1 -1  1 -1]
self.basis_table.R is an array: [2 5]
point_types: [1 0 1]
self.basis_table.R[point_types]: [5 2 5]
Cutoff: [4.9999 1.9999 4.9999]
In BasisMatrixData.from_config 1:
edge_index: [[0 1 1 2]
 [1 0 2 1]]
edge_types: [-1  1  1 -1]
In sort_edge_index:
isc_off: [[[0]]]
sc_shifts: [[0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]]
isc: [0 0 0 0]
In BasisMatrixData.from_config 2:
edge_index: [[1 0 1 2]
 [0 1 2 1]]
edge_types: [ 1 -1  1 -1]
In BasisMatrixData.from_config 3:
edge_index: [[1 0 1 2]
 [0 1 2 1]]


Calling the function
--------------------

We now have our batch object, `data`. It is a `Batch` object. In the previous notebook, we called the function from a `BasisMatrixTorchData` object. One might think that having batched data might make it more complicated to call the function.

However, it is **exactly the same code that you have to use to compute matrices in a batch**. First, of course, we need to get our inputs, which we generate artificially here (in the batch we have 6 nodes, each of them needs a scalar and a vector):

In [10]:
node_inputs = get_environment_representation(data, node_feats_irreps)
node_inputs

tensor([[ 0.7287,  0.0877,  1.0900,  1.0851],
        [-1.4684,  1.1960, -0.1784,  0.0000],
        [ 0.7287,  0.0877,  1.0900, -1.0851],
        [ 1.4734, -0.8191, -0.7607, -0.6156],
        [ 0.0604,  0.9948,  0.0930, -0.7948],
        [ 1.4734, -0.8191, -0.7607,  0.6156]])

And from them, we compute the matrices. We use the inputs as well as the preprocessed data in the batch, with exactly the same code that we have already seen:

In [11]:
node_labels, edge_labels = model(data, node_feats=node_inputs)

In Graph2Mat _get_labels_resort_index: 
BEFORE CALLING get_labels_resorting_array
types:  [0 1 0 1 0 1]
shapes:  [[1 5]
 [1 5]]
shapes_inv:  [[1 5]
 [1 5]]
transpose_neg:  False
kwargs:  {}
AFTER CALLING get_labels_resorting_array
indices:  [ 0  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25
 26 27  1 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48
 49 50 51 52  2 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71
 72 73 74 75 76 77]
In Graph2Mat _forward_interactions: 
graph2mat_edge_types:  tensor([ 1, -1,  1, -1,  1, -1,  1, -1], dtype=torch.int32)
edge_types:  tensor([ 1, -1,  1, -1,  1, -1,  1, -1], dtype=torch.int32)
In Graph2Mat _forward_interactions: 
i_edges: (graph2mat_edge_types[mask] == edge_type)  tensor([ True, False,  True, False,  True, False,  True, False])
j_edges: (~i_edges) tensor([False,  True, False,  True, False,  True, False,  True])
In Graph2Mat _forward_interactions: 
i_edges: (graph2mat_edge_types[mask] == edge_type) 

Disentangling the batch
----------------------

As simple as it is to run a batched calculation, **disentangling everything back into individual cases is harder**. It is even harder in our case, in which we have **batched sparse matrices**.

Not only you have to handle the indices of the sparsity pattern, but also the additional aggregation of the batches. This is the reason why in the `BasisMatrixData` objects you can see so many pointer arrays. They are needed to keep track of the organization.

Making use of those indices, **the data processor can disentangle the batch** and give you the individual cases. You'll be happy to see that you can call the `matrix_from_data` method of the processor, **just as you did with the single matrix case**, and it will return a `tuple` of matrices instead of just one:

In [12]:
matrices = processor.matrix_from_data(
    data,
    predictions={"node_labels": node_labels, "edge_labels": edge_labels},
)
matrices

In processing.py matrix_from_data:
data: TorchBasisMatrixDataBatch(
  metadata={ data_processor=[2] },
  edge_index=[2, 8],
  num_nodes=6,
  neigh_isc=[8],
  n_edges=[2],
  positions=[6, 3],
  shifts=[8, 3],
  cell=[6, 3],
  nsc=[2, 3],
  node_attrs=[6, 2],
  point_types=[6],
  edge_types=[8],
  batch=[6],
  ptr=[3]
)
is_batch: True
In MatrixDataProcessor.yield_from_batch:
arrays=data.numpy_arrays(): <graph2mat.core.data.processing.NumpyArraysProvider object at 0x756903653a60>
atom_ptr: [0 3 6]
edge_ptr: [0 4 8]
In BasisTableWithEdges.point_block_pointer:
  point_types = [0 1 0 1 0 1]
  point_block_size = [ 1 25]
  pointers = [ 0  1 26 27 52 53 78]
In BasisTableWithEdges.edge_block_pointer:
  edge_types = [ 1 -1  1 -1  1 -1  1 -1]
  edge_block_size = [ 1  5 25]
  pointers = [ 0  5 10 15 20 25 30 35 40]
example 0 (batch):
  atom_start: 0
  atom_end: 3
  edge_start: 0
  edge_end: 4
  new_edge_label = edge_labels[edge_labels_ptr[edge_start]: edge_labels_ptr[edge_end]]:
 this takes the edg

(<7x7 sparse array of type '<class 'numpy.float32'>'
 	with 47 stored elements in Compressed Sparse Row format>,
 <11x11 sparse array of type '<class 'numpy.float32'>'
 	with 71 stored elements in Compressed Sparse Row format>)

<div class="alert alert-info">

Note

`matrix_from_data` has automatically detected that the data passed was a `torch_geometric`'s `Batch` object. There's also the `is_batch` argument to explicitly indicate if it is a batch or not. Also, the processor has the `yield_from_batch` method, which is more explicit and will return a generator instead of a tuple, which is better for very big matrices if you want to process them individually.

</div>

As we already did in the previous notebook, we can plot the matrices:

In [13]:
for config, matrix in zip(configs, matrices):
    plot_basis_matrix(
        matrix,
        config,
        point_lines={"color": "black"},
        basis_lines={"color": "blue"},
        colorscale="temps",
        text=".2f",
        basis_labels=True,
    ).show()

Try to relate the matrices to the systems we created and see if their shape makes sense :)

Summary and next steps
-----------

In this notebook we learned **how to batch systems** and then **use the data processor to unbatch them**.

The **next steps** could be:

- Understanding how to **train the function** to produce the target matrix. See [this notebook](<./Fitting matrices.ipynb>).
- Combining this function with other modules for your particular application.